# Study 2: joint interferometer and Thomson placement

Reads the scan written by `run_scan.py`: in each of study 1's three best planes, every laser (five directions, five parallel offsets) with every admissible pair of optic surface angles (two per optic), one chord on the laser and the second chord picked from study 1's catalogue, scored by the expected Kullback-Leibler divergence about the `ne` and `Te` parameters (bits, mean over the eight truths). The ten volumes sit evenly along the laser's plasma span and are not optimized. Figures go to `plots/` as PDF, tables to `tables/`.

In [ ]:
import functools
import json

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from plasma_state_estimation_for_diagnostic_placement_optimization.bayesian_inference import (  # noqa: E501
    desc_truth_types,
    inference_types,
    profile_parametrization_types,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.diagnostics_forward_models import (  # noqa: E501
    geometry_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.diagnostics_forward_models import (  # noqa: E501
    thomson_scattering_utils as ts,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.diagnostics_placement_optimization import (  # noqa: E501
    diagnostics_placement_optimization_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.ideal_mhd import (
    desc_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.mhd_equilibrium_forward_models import (  # noqa: E501
    dataset_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.results.diagnostics_placement_optimization import (  # noqa: E501
    run_placement_inference,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.results.diagnostics_placement_optimization.if_and_ts_placement_study import (  # noqa: E501
    run_scan,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    inference_store_utils,
    parametrization_fit_utils,
    profile_table_utils,
    pubfig,
    sweep_utils,
    synthetic_target_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    plots_plasma_state_estimation as plots,
)

pubfig.apply_style()

SCAN = run_scan.OUTPUT
STUDY = SCAN.parents[1]
PLOTS = STUDY / "plots"
PLOTS.mkdir(exist_ok=True)
PACKAGE = SCAN.parents[5]
DATA = PACKAGE.parents[1] / "data"
PARQUET = DATA / f"{dataset_utils.DEFAULT_DATASET}.parquet"
SETTINGS = DATA / f"{dataset_utils.DEFAULT_DATASET}_settings.json"
PATHS = sweep_utils.SweepPaths(
    data_directory=DATA,
    dataset=dataset_utils.DEFAULT_DATASET,
    models_directory=PACKAGE / "mhd_equilibrium_forward_models" / "trained_models",
    model_name=inference_types.DEFAULT_SURROGATE_NAME,
)
TRUTH_NAMES = diagnostics_placement_optimization_utils.TRUTH_NAMES
LAYOUT_COLUMNS = [
    "laser_surface_angles",
    "optic_surface_angles",
    "volume_span_fractions",
    "chord_surface_angles",
]

reference = sweep_utils.reference_truth(PARQUET, SETTINGS)
table = pd.read_csv(SCAN).sort_values(
    "information_gain_bits", ascending=False, ignore_index=True
)
print(
    f"{len(table)} layouts in {table.toroidal_angle_deg.nunique()} planes, "
    f"{table.information_gain_bits.min():.2f} to "
    f"{table.information_gain_bits.max():.2f} bit"
)
table.drop(columns=LAYOUT_COLUMNS).head()

## 1. Gain of every layout

One marker per layout (laser offset and optic pair) against its laser direction, one panel per scanned plane, the plane's best layout filled. A laser direction without markers had no offset whose volumes every optic could serve inside its scattering-angle window.

In [ ]:
planes = sorted(table.toroidal_angle_deg.unique())
per_laser = table.groupby(
    ["toroidal_angle_deg", "laser_direction_deg", "laser_offset_m"]
).information_gain_bits
optic_spread = per_laser.max() - per_laser.min()
print(
    f"the optic pair moves the gain by {optic_spread.median():.2f} bit (median), "
    f"{optic_spread.max():.2f} bit at most"
)
figure, axes = pubfig.subplots(1, len(planes), width="double", aspect=0.9, sharey=True)
for axis, plane in zip(axes, planes):
    rows = table[table.toroidal_angle_deg == plane]
    axis.plot(
        rows.laser_direction_deg,
        rows.information_gain_bits,
        "o",
        mfc="none",
        color=pubfig.PALETTE[0],
        ms=4,
        label="layout",
    )
    best = rows.loc[rows.information_gain_bits.idxmax()]
    axis.plot(
        best.laser_direction_deg,
        best.information_gain_bits,
        "o",
        color=pubfig.PALETTE[1],
        ms=5,
        label="best of the plane",
    )
    axis.set_title(
        f"{plane:.1f} deg, best {best.information_gain_bits:.2f} bit", fontsize="small"
    )
    axis.set_xlabel("laser direction [deg]")
    axis.set_xticks(run_scan.LASER_DIRECTIONS_DEG)
axes[0].set_ylabel("information gain [bit]")
handles, labels = axes[0].get_legend_handles_labels()
figure.legend(handles, labels, loc="outside lower center", ncols=2)
pubfig.save(figure, str(PLOTS / "if_and_ts_placement_gain_per_layout"))

## 2. Initial guess against the optimized layout

- **Initial guess**: nothing optimized. Study 1's first plane, the horizontal laser nearest the plasma centre with its first admissible optic pair, one chord on the laser and study 1's vertical chord through the middle of the plasma, scored like a scan row (`run_scan.initial_guess`, a few minutes: it loads the surrogate and scores the plane's chord catalogue).
- **Optimized**: the best row of the scan.

Both carry ten volumes and two chords. The table lists both layouts with their gain per truth.

In [ ]:
layouts = run_scan.compared_layouts(table, PATHS)
TITLES = {"initial_guess": "initial guess", "optimized": "optimized"}
pd.DataFrame(
    {title: layouts[layout].drop(LAYOUT_COLUMNS) for layout, title in TITLES.items()}
)

In [ ]:
def draw_layout(axis, row, title):
    toroidal_angle = np.radians(row.toroidal_angle_deg)
    plasma, surface = geometry_utils.cross_section_and_surface(
        reference.coefficients, toroidal_angle, geometry_utils.DEFAULT_SURFACE_OFFSET
    )
    axis.plot(*np.asarray(plasma), "k-", lw=1, label="plasma boundary")
    axis.plot(
        *np.asarray(surface.curve), color="0.6", lw=1, label="auxiliary design surface"
    )
    for angles in json.loads(row.chord_surface_angles):
        ends = np.asarray(
            geometry_utils.surface_point_at_angle(surface, np.asarray(angles))
        )
        axis.plot(ends[0], ends[1], "r-", lw=1, label="interferometer chords")
    laser = json.loads(row.laser_surface_angles)
    ends = np.asarray(geometry_utils.surface_point_at_angle(surface, np.asarray(laser)))
    axis.plot(ends[0], ends[1], "g-", lw=1, label="Thomson scattering laser")
    volumes, _, _ = geometry_utils.thomson_volumes_on_chord(
        reference.coefficients,
        toroidal_angle,
        geometry_utils.DEFAULT_SURFACE_OFFSET,
        tuple(laser),
        0.0,
        tuple(json.loads(row.volume_span_fractions)),
    )
    axis.plot(volumes[0], volumes[1], "go", ms=4, label="scattering volumes")
    optics = np.asarray(
        geometry_utils.surface_point_at_angle(
            surface, np.asarray(json.loads(row.optic_surface_angles))
        )
    )
    axis.plot(optics[0], optics[1], "b^", ms=8, label="collection optics")
    magnetic_axis = geometry_utils.magnetic_axis(reference.coefficients, toroidal_angle)
    axis.plot(*magnetic_axis, "kx", ms=7, mew=1.5, label="magnetic axis")
    axis.set_aspect("equal")
    axis.set_title(
        f"{title}\n{row.toroidal_angle_deg:.1f} deg, "
        f"{row.information_gain_bits:.2f} bit",
        fontsize=9,
    )
    axis.set_xlabel("R [m]")
    axis.set_ylabel("Z [m]")


figure, axes = plt.subplots(1, 2, figsize=(9, 4.6), layout="constrained")
for axis, (layout, title) in zip(axes, TITLES.items()):
    draw_layout(axis, layouts[layout], title)
handles, labels = axes[0].get_legend_handles_labels()
# Each chord is its own line; one legend entry per label.
legend = dict(zip(labels, handles))
figure.legend(legend.values(), legend.keys(), loc="outside lower center", ncols=4)
figure.savefig(PLOTS / "if_and_ts_placement_initial_guess_vs_optimized.pdf")

## 3. Inference on the two layouts

`map_laplace` on both layouts and NUTS on the optimized one, stored by the shared placement runner one folder up,

```
python run_placement_inference.py --study if_and_ts_placement_study --workers 4
python run_placement_inference.py --study if_and_ts_placement_study --method mcmc_nuts --layouts optimized --workers 4
python run_placement_inference.py --study if_and_ts_placement_study --layouts initial_guess optimized separately_optimized sequentially_optimized --n-seeds 5 --skip-stored --workers 4
```

with five noise draws per truth for `map_laplace` on the layouts of sections 3 and 5 (the last command adds draws 1 to 4) and one for NUTS, each draw shared by every layout and method of a truth, under the same band rule and noise model. The tables average each truth over its draws before the standard error over the truths; the figures show draw 0. **Initial guess** against **optimized** is what optimizing the laser, its optics and the chords together buys the reconstruction; NUTS against the Laplace band (section 3.3) checks the Gaussian approximation the design criterion rests on.

One figure per truth and layout, drawn as in `mcmc_benchmark`: the truth, the prior draw the MAP started from, the MAP with its Laplace 5 to 95 percent band, and each volume's own `ne` and `Te` from its channels alone.

In [ ]:
run_ids = sweep_utils.current_ids(run_placement_inference.ids_file(STUDY.name))
assert run_ids, (
    "no stored runs yet: run `python run_placement_inference.py --study "
    f"{STUDY.name} --workers 4` one folder up first"
)
records = {
    label: inference_store_utils.load_record(
        inference_store_utils.INFERENCE_STORE, identifier
    )
    for label, identifier in run_ids.items()
}
truths = {
    name: desc_truth_types.read_desc_truth(name, PARQUET, SETTINGS)
    for name in TRUTH_NAMES
}
thomson_points = {}
for truth_name, truth in truths.items():
    for layout, title in TITLES.items():
        label = run_placement_inference.run_label(layout, truth_name)
        thomson_points[label] = profile_table_utils.thomson_points_table(
            records[label], truth
        )
        figure = plots.plot_method_band_comparison(
            profile_table_utils.profile_band_table(records[label], "map_laplace"),
            ("map_laplace",),
            thomson_points=thomson_points[label],
            title=f"{truth_name}, {title}",
        )
        pubfig.save(
            figure, str(PLOTS / f"if_and_ts_placement_profiles_{truth_name}_{layout}")
        )
        display(figure)
        plt.close(figure)

### 3.1 Did the MAP converge?

Per truth and layout: chi-squared per degree of freedom at the MAP, the converged starts (a run draws a new start from the priors until one converges) and the L-BFGS iterations. A chi-squared per degree of freedom far above one at a converged start is a local minimum, which the figure above shows as a profile away from the truth.

In [ ]:
rows = []
for truth_name in TRUTH_NAMES:
    for layout, title in TITLES.items():
        record = records[run_placement_inference.run_label(layout, truth_name)]
        rows.append(
            {
                "layout": title,
                "truth": truth_name,
                "chi2 / dof": record.chi_squared / record.n_degrees_of_freedom,
                "converged starts": record.n_map_starts_converged,
                "iterations": record.map_iterations,
                "time [s]": record.map_time_seconds,
            }
        )
pd.DataFrame(rows).set_index(["layout", "truth"]).round(
    {"chi2 / dof": 1, "time [s]": 0}
)

### 3.2 Reconstruction accuracy: initial guess against optimized

Per truth and layout, `map_laplace` only (`run_placement_inference.accuracy_table`, shared with study 1): the RMS error of the reported mean in percent of the truth's peak and the median half-width of the 5 to 95 percent band in percent of the local value, for `ne` and `Te`, with whether the MAP converged and the information gain of its Laplace Gaussian. The summary (`accuracy_summary`) averages each truth over its draws, then over the eight truths, with the standard error between them. Both go to `tables/`.

In [ ]:
per_run = run_placement_inference.accuracy_table(run_ids, list(TITLES))
per_run.to_csv(SCAN.parent / "if_and_ts_placement_accuracy_per_run.csv", index=False)
# The per-run table keeps the unconverged draws, flagged; every mean leaves them out.
per_run = per_run[per_run.map_converged]
summary = run_placement_inference.accuracy_summary(per_run)
summary.to_csv(SCAN.parent / "if_and_ts_placement_accuracy_summary.csv")
display(
    per_run.pivot_table(
        index=["layout", "truth_name"],
        columns="profile",
        values=list(run_placement_inference.METRICS),
        sort=False,
    ).round(2)
)
summary

### 3.3 NUTS check of the Laplace band on the optimized layout

NUTS started from the MAP, on the same data as the `map_laplace` run of the same truth, with the sampler of `mcmc_benchmark`. The table gives its convergence diagnostics and `nuts_converged`, the sampler's own verdict, next to the median band half-width of each method for `ne` and `Te`; read a truth's widths only where it converged. The figures draw both bands over the truth, as section 3 of `mcmc_benchmark` does.

In [ ]:
"""WIDTH_PROFILES = {"ne": "electron_density", "Te": "electron_temperature"}
rows = []
for truth_name in TRUTH_NAMES:
    laplace = records[run_placement_inference.run_label("optimized", truth_name)]
    nuts = records[
        run_placement_inference.run_label("optimized", truth_name, "mcmc_nuts")
    ]
    widths = {
        (method, accuracy.name): accuracy.uncertainty_percent
        for method, record in (("Laplace", laplace), ("NUTS", nuts))
        for accuracy in inference_metrics_utils.profile_accuracy(record)
    }
    rows.append(
        {
            "truth": truth_name,
            "max r_hat": nuts.r_hat_max,
            "min bulk ESS": nuts.ess_bulk_min,
            "min tail ESS": nuts.ess_tail_min,
            "divergences": nuts.n_divergences,
            "tree depth cap": nuts.max_tree_depth_fraction,
            "converged": nuts.nuts_converged,
            **{
                f"{symbol} half-width {method} [%]": widths[(method, name)]
                for symbol, name in WIDTH_PROFILES.items()
                for method in ("Laplace", "NUTS")
            },
        }
    )
    figure = plots.plot_method_band_comparison(
        pd.concat(
            [
                profile_table_utils.profile_band_table(laplace, "map_laplace"),
                profile_table_utils.profile_band_table(nuts, "mcmc_nuts"),
            ],
            ignore_index=True,
        ),
        ("map_laplace", "mcmc_nuts"),
        thomson_points=thomson_points[
            run_placement_inference.run_label("optimized", truth_name)
        ],
        title=f"{truth_name}, optimized",
    )
    pubfig.save(
        figure, str(PLOTS / f"if_and_ts_placement_profiles_{truth_name}_optimized_nuts")
    )
    display(figure)
    plt.close(figure)
nuts_check = pd.DataFrame(rows).set_index("truth")
nuts_check.to_csv(SCAN.parent / "if_and_ts_placement_nuts_check.csv")
nuts_check.round({"max r_hat": 3, "min bulk ESS": 0, "min tail ESS": 0}).round(2)"""

## 4. Why a layout resolves `Te` better or worse: the Cramer-Rao bound of every volume

The Cramer-Rao bound on `Te` of every scattering volume, at the `ne` and `Te` each of the eight plasma states takes there, with `ne` free so that only the channel ratios inform `Te`. The first figure plots it against the volume's `Te` for both layouts: the volumes of each optic form a branch, the back-scattering one rising with `Te` because its wider spectrum leaves the passband range, and the layouts differ in how many volumes lie on each branch. The CSV sums the information of the ten volumes per plasma state into one bound, initial guess against optimized, the table of the thesis. The last figure is the bound against `Te` of one volume per optic at a fixed density, a guide to the branches.


In [ ]:
placements = {
    title: synthetic_target_utils.build_diagnostic_placement(
        run_scan.design_point(layouts[layout]), reference, verbose=False
    )
    for layout, title in TITLES.items()
}
scattering_angles_degrees = {}
first_volume_per_optic = {}
for title, placement in placements.items():
    geometry = placement.thomson_geometry
    scattering_angles_degrees[title] = np.degrees(
        np.asarray(ts.volume_scattering_angles(geometry))
    )
    print(
        f"{title}: scattering angles "
        f"{np.round(scattering_angles_degrees[title], 1)} deg"
    )
    # In optic order, skipping an optic that serves no volume.
    optic_index = np.asarray(geometry.volume_optic_index)
    first_volume_per_optic[title] = [
        int(np.flatnonzero(optic_index == optic)[0])
        for optic in range(np.asarray(geometry.optic_positions).shape[0])
        if np.any(optic_index == optic)
    ]

# Te of every volume each optic serves, over the eight plasma states, in keV.
truth_profiles = [
    desc_truth_types.read_desc_truth(name, PARQUET, SETTINGS).profiles
    for name in TRUTH_NAMES
]
volume_temperatures_kev = {}
for title, placement in placements.items():
    geometry = placement.thomson_geometry
    rho = desc_utils.xyz_to_flux_coordinates(
        reference.coefficients, jnp.asarray(geometry.volume_positions)
    ).rho
    optic_index = np.asarray(geometry.volume_optic_index)
    volume_temperatures_kev[title] = [
        np.concatenate(
            [
                np.asarray(
                    profile_parametrization_types.electron_temperature(
                        profiles, rho[optic_index == optic]
                    )
                )
                / 1e3
                for profiles in truth_profiles
            ]
        )
        for optic in range(np.asarray(geometry.optic_positions).shape[0])
        if np.any(optic_index == optic)
    ]

TEMPERATURES_EV = np.geomspace(20.0, 1.0e4, 200)
SENSITIVITY_DENSITY = 3.0e19  # m^-3, a core-like density for every curve
temperature_grid = jnp.asarray(TEMPERATURES_EV, dtype=float)
TEMPERATURE_BINS_KEV = np.linspace(0.0, 10.0, 21)
OPTIC_COLORS = {"forward": pubfig.PALETTE[0], "back": pubfig.PALETTE[1]}

# Cramer-Rao uncertainty of Te of every volume at its own ne and Te, over the eight
# plasma states; ne is a free parameter, so only the channel ratios inform Te.
rows = []
for title, placement in placements.items():
    geometry = placement.thomson_geometry
    rho = desc_utils.xyz_to_flux_coordinates(
        reference.coefficients, jnp.asarray(geometry.volume_positions)
    ).rho
    variance = functools.partial(
        ts.log_temperature_variance,
        geometry=geometry,
        response=placement.polychromator_response,
        calibration=ts.absolute_calibration(geometry),
        noise_model=placement.design.thomson_noise_model,
        signal_scale=placement.thomson_signal_scale,
    )
    for truth_name, profiles in zip(TRUTH_NAMES, truth_profiles):
        density = profile_parametrization_types.electron_density(profiles, rho)
        temperature = profile_parametrization_types.electron_temperature(profiles, rho)
        sigma = jnp.sqrt(variance(jnp.stack([jnp.log(density), jnp.log(temperature)])))
        for volume, angle in enumerate(scattering_angles_degrees[title]):
            rows.append(
                {
                    "layout": title,
                    "truth": truth_name,
                    "volume": volume,
                    "optic": "forward" if angle < 90.0 else "back",
                    "scattering_angle_deg": float(angle),
                    "temperature_kev": float(temperature[volume]) / 1e3,
                    "relative_uncertainty_percent": 100.0 * float(sigma[volume]),
                }
            )
cramer_rao = pd.DataFrame(rows)
cramer_rao.round(4).to_csv(
    SCAN.parent / "if_and_ts_placement_temperature_cramer_rao.csv", index=False
)
display(
    cramer_rao.groupby(["layout", "optic"])
    .agg(
        volume_temperatures=("relative_uncertainty_percent", "size"),
        median_percent=("relative_uncertainty_percent", "median"),
        scattering_angle_min=("scattering_angle_deg", "min"),
        scattering_angle_max=("scattering_angle_deg", "max"),
    )
    .round(2)
)

# Te information summed over the ten volumes of each plasma state, and its share
# per optic, to quote alongside the channel signals.
cramer_rao["information"] = 1.0 / np.square(
    cramer_rao.relative_uncertainty_percent / 100.0
)
combined = 100.0 / np.sqrt(
    cramer_rao.groupby(["layout", "truth"]).information.sum().unstack("layout")
)
print("combined Cramer-Rao sigma_Te / Te [%], mean over the plasma states:")
print(combined.mean().round(2).to_string())
print("information share per optic:")
print(
    (
        cramer_rao.groupby(["layout", "optic"]).information.sum()
        / cramer_rao.groupby("layout").information.sum()
    )
    .round(3)
    .to_string()
)

# The bound of every volume against its Te, one panel per layout: the volumes of each
# optic form a branch, and the layouts differ in how many volumes lie on each.
# The dashed guide is the bound of the volume of median angle of each optic at a fixed,
# state-like density; each of its steps is a further channel rising above the noise floor
# of the electronic model.
GUIDE_DENSITY = 5.0e20  # m^-3
guide_temperatures = jnp.asarray(np.geomspace(150.0, 1.2e4, 300))
figure, axes = pubfig.subplots(1, 2, width="double", aspect=0.8, sharey=True)
for axis, (title, placement) in zip(axes.flat, placements.items()):
    geometry = placement.thomson_geometry
    n_volume = np.asarray(geometry.volume_positions).shape[0]
    guide_variance = jax.vmap(
        functools.partial(
            ts.log_temperature_variance,
            geometry=geometry,
            response=placement.polychromator_response,
            calibration=ts.absolute_calibration(geometry),
            noise_model=placement.design.thomson_noise_model,
            signal_scale=placement.thomson_signal_scale,
        )
    )(
        jnp.stack(
            [
                jnp.full((guide_temperatures.size, n_volume), jnp.log(GUIDE_DENSITY)),
                jnp.repeat(jnp.log(guide_temperatures)[:, None], n_volume, axis=1),
            ],
            axis=1,
        )
    )
    angles = scattering_angles_degrees[title]
    for optic, part in cramer_rao[cramer_rao.layout == title].groupby("optic"):
        axis.scatter(
            part.temperature_kev,
            part.relative_uncertainty_percent,
            s=16,
            lw=0.8,
            color=OPTIC_COLORS[optic],
            label=f"{optic} optic, {part.volume.nunique()} volumes",
            zorder=3,
        )
        members = np.flatnonzero((angles < 90.0) == (optic == "forward"))
        median_volume = int(members[np.argsort(angles[members])[(members.size - 1) // 2]])
        axis.plot(
            np.asarray(guide_temperatures) / 1e3,
            100.0 * np.sqrt(np.asarray(guide_variance[:, median_volume])),
            color=OPTIC_COLORS[optic],
            lw=1.0,
            ls="--",
            label=rf"$\theta = {angles[median_volume]:.0f}^\circ$ at $n_e = 5\times10^{{20}}$ m$^{{-3}}$",
        )
    axis.set_xscale("log")
    axis.set_yscale("log")
    axis.set_xlim(0.15, 12.0)
    axis.set_ylim(0.8, 40.0)
    axis.set_xlabel(r"$T_e$ of the volume [keV]")
    axis.set_title(f"{title}: one point per volume and plasma state", fontsize="small")
    axis.legend(fontsize="x-small", loc="upper left")
axes.flat[0].set_ylabel(r"Cramer-Rao bound $\sigma_{T_e}/T_e$ [%]")
pubfig.save(figure, str(PLOTS / "if_and_ts_placement_temperature_cramer_rao_per_volume"))

# The combined bound of the ten volumes per plasma state, the table of the thesis; the
# mean row averages every column over the plasma states, the differences included.
per_state = combined.copy()
per_state["difference_percent"] = (
    100.0 * (per_state["optimized"] - per_state["initial guess"]) / per_state["initial guess"]
)
per_state.loc["mean"] = per_state.mean()
per_state.round(3).to_csv(
    SCAN.parent / "if_and_ts_placement_temperature_cramer_rao_per_state.csv"
)
display(per_state.round(3))

# Cramer-Rao relative uncertainty of Te, one curve per layout and optic.
figure, axes = pubfig.subplots(1, 1, width="single", aspect=0.8)
axis = axes.flat[0]
line_styles = ("-", "--", ":")
for index, (title, placement) in enumerate(placements.items()):
    geometry = placement.thomson_geometry
    n_volume = np.asarray(geometry.volume_positions).shape[0]
    log_states = jnp.stack(
        [
            jnp.full((temperature_grid.size, n_volume), jnp.log(SENSITIVITY_DENSITY)),
            jnp.repeat(jnp.log(temperature_grid)[:, None], n_volume, axis=1),
        ],
        axis=1,
    )
    variance = jax.vmap(
        functools.partial(
            ts.log_temperature_variance,
            geometry=geometry,
            response=placement.polychromator_response,
            calibration=ts.absolute_calibration(geometry),
            noise_model=placement.design.thomson_noise_model,
            signal_scale=placement.thomson_signal_scale,
        )
    )(log_states)
    for optic, volume in enumerate(first_volume_per_optic[title]):
        axis.plot(
            np.asarray(temperature_grid) / 1e3,
            100.0 * np.sqrt(np.asarray(variance[:, volume])),
            color=pubfig.CATEGORICAL_PALETTE[index % len(pubfig.CATEGORICAL_PALETTE)],
            ls=line_styles[optic % len(line_styles)],
            lw=1.2,
            label=rf"{title}, "
            rf"$\theta = {scattering_angles_degrees[title][volume]:.0f}^\circ$",
        )
axis.set_xscale("log")
axis.set_yscale("log")
axis.set_xlabel(r"$T_e$ [keV]")
axis.set_ylabel(r"$\sigma_{T_e} / T_e$ [%]")
axis.legend(fontsize="x-small")
pubfig.save(figure, str(PLOTS / "if_and_ts_placement_temperature_sensitivity"))

## 5. Joint against separate and sequential optimization

What optimizing the two diagnostics together buys, on one design space: study 1's three best planes, a first chord through the magnetic axis in one of the five laser directions, a second chord from study 1's catalogue, and the Thomson system with its laser through the axis and its optics spread edge to edge.

- **separate**: each diagnostic optimized alone, each in its own plane. The Thomson system by `run_scan.py --laser-through-axis --thomson-only` (scored on `ne` and `Te` without chords) and the chords by `run_scan.py --laser-through-axis --interferometer-only` (scored on `ne` alone without the Thomson system, study 1's objective), combined by `run_scan.separately_optimized`.
- **sequential**: the Thomson system optimized alone, then the chords placed in its plane with the first one along its laser by the interferometer-only objective (`run_scan.sequentially_optimized`, the interferometer-only row on the Thomson optimum's plane and laser).
- **joint**: the optimum of section 2.

All three carry ten volumes and two chords, and all are scored and inferred with both diagnostics.

```
python run_scan.py --laser-through-axis --thomson-only
python run_scan.py --laser-through-axis --interferometer-only
python run_placement_inference.py --study if_and_ts_placement_study --layouts separately_optimized sequentially_optimized --workers 4
```

`comparison_table` gives, per layout, the mean and standard error over the plasma states, each averaged over its noise draws, of the expected gain, the gain realized at the MAP and the RMSE and band of `ne` and `Te`, as in the thesis table of section 3.2, with the difference of the last layout relative to each of the others.

In [ ]:
def comparison_table(compared, truth_names, gains):
    per_run = run_placement_inference.accuracy_table(
        run_ids, list(compared), truth_names
    )
    unconverged = per_run[~per_run.map_converged].drop_duplicates(
        ["layout", "truth_name", "draw"]
    )
    if len(unconverged):
        print(
            "MAP not converged, left out:",
            list(zip(unconverged.layout, unconverged.truth_name, unconverged.draw)),
        )
    per_run = run_placement_inference.state_means(per_run[per_run.map_converged])
    print(
        "noisy draws per plasma state:",
        {
            layout: sorted(set(map(int, draws)))
            for layout, draws in per_run.groupby("layout", sort=False).n_draws
        },
    )
    columns, means = {}, {}
    for layout, title in compared.items():
        runs = per_run[per_run.layout == layout]
        density = runs[runs.profile == "electron_density"]
        temperature = runs[runs.profile == "electron_temperature"]
        values = {
            "EIG [bit]": gains.loc[layout, [f"bits_{name}" for name in truth_names]],
            "MAP gain [bit]": density.information_gain_bits,
            "ne RMSE [%]": density.error_percent,
            "ne band [%]": density.uncertainty_percent,
            "Te RMSE [%]": temperature.error_percent,
            "Te band [%]": temperature.uncertainty_percent,
        }
        values = {metric: np.asarray(v, dtype=float) for metric, v in values.items()}
        means[title] = {metric: v.mean() for metric, v in values.items()}
        columns[title] = {
            metric: f"{v.mean():.2f} +- {v.std(ddof=1) / np.sqrt(v.size):.2f}"
            for metric, v in values.items()
        }
    # One difference column between two layouts, else the last against each other one.
    titles = list(means)
    pairs = (
        {"difference [%]": (titles[0], titles[1])}
        if len(titles) == 2
        else {
            f"{titles[-1]} vs {earlier} [%]": (earlier, titles[-1])
            for earlier in titles[:-1]
        }
    )
    for name, (earlier, later) in pairs.items():
        ratio = {m: means[later][m] / means[earlier][m] for m in means[earlier]}
        columns[name] = {m: f"{100.0 * (r - 1.0):+.1f}" for m, r in ratio.items()}
    return pd.DataFrame(columns)


thomson_table = pd.read_csv(run_scan.THOMSON_ONLY_OUTPUT)
chord_table = pd.read_csv(run_scan.INTERFEROMETER_ONLY_OUTPUT)
print(
    f"Thomson alone: {len(thomson_table)} layouts, "
    f"{thomson_table.information_gain_bits.min():.2f} to "
    f"{thomson_table.information_gain_bits.max():.2f} bit without chords"
)
print(
    f"Chords alone: {len(chord_table)} axis chords, "
    f"{chord_table.information_gain_bits.min():.2f} to "
    f"{chord_table.information_gain_bits.max():.2f} bit on ne without Thomson"
)
SEPARATE_TITLES = {
    "separately_optimized": "separate",
    "sequentially_optimized": "sequential",
    "optimized": "joint",
}
pd.DataFrame(
    {
        title: layouts[layout].drop(LAYOUT_COLUMNS)
        for layout, title in SEPARATE_TITLES.items()
    }
)

In [ ]:
separate_comparison = comparison_table(
    SEPARATE_TITLES, TRUTH_NAMES, pd.DataFrame(layouts).T
)
separate_comparison.to_csv(SCAN.parent / "if_and_ts_placement_separate_comparison.csv")
separate_comparison

In [ ]:
def draw_chords(axis, toroidal_angle_deg, chord_surface_angles):
    toroidal_angle = np.radians(toroidal_angle_deg)
    plasma, surface = geometry_utils.cross_section_and_surface(
        reference.coefficients, toroidal_angle, geometry_utils.DEFAULT_SURFACE_OFFSET
    )
    axis.plot(*np.asarray(plasma), "k-", lw=1, label="plasma boundary")
    axis.plot(
        *np.asarray(surface.curve), color="0.6", lw=1, label="auxiliary design surface"
    )
    for angles in json.loads(chord_surface_angles):
        ends = np.asarray(
            geometry_utils.surface_point_at_angle(surface, np.asarray(angles))
        )
        axis.plot(ends[0], ends[1], "r-", lw=1, label="interferometer chords")
    magnetic_axis = geometry_utils.magnetic_axis(reference.coefficients, toroidal_angle)
    axis.plot(*magnetic_axis, "kx", ms=7, mew=1.5, label="magnetic axis")
    axis.set_aspect("equal")
    axis.set_xlabel("R [m]")
    axis.set_ylabel("Z [m]")


# The separate layout keeps its chords in their own plane, so it takes two panels:
# its Thomson system without chords, and its chords in the interferometer plane.
separate, sequential, joint = (layouts[layout] for layout in SEPARATE_TITLES)
separate_thomson = separate.copy()
separate_thomson["chord_surface_angles"] = "[]"
figure, axes = plt.subplots(2, 2, figsize=(7.5, 8.2), layout="constrained")
draw_layout(axes[0, 0], separate_thomson, "")
draw_chords(
    axes[0, 1],
    separate.interferometer_toroidal_angle_deg,
    separate.chord_surface_angles,
)
draw_layout(axes[1, 0], sequential, "")
draw_layout(axes[1, 1], joint, "")
titles = (
    f"separate ({separate.information_gain_bits:.2f} bit)\n"
    f"Thomson scattering, {separate.toroidal_angle_deg:.1f} deg",
    f"separate ({separate.information_gain_bits:.2f} bit)\n"
    f"interferometer, {separate.interferometer_toroidal_angle_deg:.1f} deg",
    f"sequential ({sequential.information_gain_bits:.2f} bit)\n"
    f"{sequential.toroidal_angle_deg:.1f} deg",
    f"joint ({joint.information_gain_bits:.2f} bit)\n"
    f"{joint.toroidal_angle_deg:.1f} deg",
)
for axis, title in zip(axes.flat, titles):
    axis.set_title(title, fontsize=9)
# Each chord is its own line and no panel shows every element; one entry per label.
legend = {
    label: handle
    for axis in axes.flat
    for handle, label in zip(*axis.get_legend_handles_labels())
}
figure.legend(legend.values(), legend.keys(), loc="outside lower center", ncols=4)
figure.savefig(PLOTS / "if_and_ts_placement_separate_vs_joint.pdf")

Where the joint layout's narrower `ne` band comes from. For each layout and plasma state, the Laplace gain on the `ne` parameters as reported (the calibrations `delta_j` marginalized) and with the calibrations known (the Gaussian conditioned on them), and the relative posterior standard deviation of `n_e0` from the stored samples. If the gain came from the Thomson calibration, knowing the `delta_j` would close the gap between the layouts. The last line scores the Thomson optimum with the second chord that the joint objective picks, the joint-scan row with the same plane, laser and optics.

In [ ]:
def information_bits(prior_covariance, posterior_covariance):
    """Gain from a prior to a posterior Gaussian, in bits."""
    return (
        0.5
        * (
            np.linalg.slogdet(prior_covariance)[1]
            - np.linalg.slogdet(posterior_covariance)[1]
        )
        / np.log(2.0)
    )


rows = []
for layout, title in SEPARATE_TITLES.items():
    for truth_name in TRUTH_NAMES:
        _, arrays = inference_store_utils.load_inference(
            inference_store_utils.INFERENCE_STORE,
            run_ids[f"{truth_name}_{layout}_seed00"],
        )
        names = list(arrays["trajectory_site_names"])
        density = [i for i, name in enumerate(names) if name.startswith("ne_")]
        calibration = [
            i for i, name in enumerate(names) if name.startswith("calibration")
        ]
        covariance = arrays["laplace_covariance"]
        prior = np.linalg.inv(arrays["laplace_prior_precision"])[
            np.ix_(density, density)
        ]
        marginal = covariance[np.ix_(density, density)]
        cross = covariance[np.ix_(density, calibration)]
        conditioned = marginal - cross @ np.linalg.solve(
            covariance[np.ix_(calibration, calibration)], cross.T
        )
        axis_density = arrays["electron_density_samples"][:, 0]
        rows.append(
            {
                "layout": title,
                "truth": truth_name,
                "ne gain [bit]": information_bits(prior, marginal),
                "ne gain, calibration known [bit]": information_bits(
                    prior, conditioned
                ),
                "ne0 relative std": axis_density.std() / abs(axis_density.mean()),
            }
        )
calibration_check = pd.DataFrame(rows)
calibration_check["known calibration adds [bit]"] = (
    calibration_check["ne gain, calibration known [bit]"]
    - calibration_check["ne gain [bit]"]
)
display(
    calibration_check.groupby("layout", sort=False)[
        ["ne gain [bit]", "known calibration adds [bit]", "ne0 relative std"]
    ]
    .agg(["mean", "max"])
    .round(3)
)
spread = calibration_check.pivot(
    index="truth", columns="layout", values="ne0 relative std"
)
for title in ("separate", "sequential"):
    ratio = spread[title] / spread["joint"]
    print(
        f"ne0 relative std, {title} / joint: {ratio.min():.2f} to {ratio.max():.2f}, "
        f"larger on {(ratio > 1.0).sum()} of {len(ratio)} states"
    )

thomson_best = thomson_table.loc[thomson_table.information_gain_bits.idxmax()]
same_thomson = table[
    np.isclose(table.toroidal_angle_deg, thomson_best.toroidal_angle_deg)
    & np.isclose(table.laser_direction_deg, thomson_best.laser_direction_deg)
    & np.isclose(table.laser_offset_m, thomson_best.laser_offset_m)
    & (table.optic_surface_angles == thomson_best.optic_surface_angles)
]
print(
    "Thomson optimum with the joint objective's second chord: "
    f"{same_thomson.information_gain_bits.max():.2f} bit, joint optimum "
    f"{table.information_gain_bits.max():.2f} bit"
)

## 6. Plasma states outside the optimization set

The initial guess and the optimized layout of section 2, scored and inferred on the eight test-split states of `desc_truth_types.TEST_STATE_IDS`, on which no layout was optimized: does the optimized layout also raise the gain and lower the error and band there?

```
python run_placement_inference.py --study if_and_ts_placement_study --layouts initial_guess optimized --truths test_state_1_parametric test_state_2_parametric test_state_3_parametric test_state_4_parametric test_state_5_gp test_state_6_gp test_state_7_gp test_state_8_gp --workers 4
```

In [ ]:
TEST_STATES = tuple(desc_truth_types.TEST_STATE_IDS)
test_gains = run_scan.layout_gains(
    PATHS, {layout: layouts[layout] for layout in TITLES}, TEST_STATES
)
display(test_gains.T.round(2))
test_comparison = comparison_table(TITLES, TEST_STATES, test_gains)
test_comparison.to_csv(SCAN.parent / "if_and_ts_placement_test_states_comparison.csv")
test_comparison

In [ ]:
# The two-power fit floor: the MAP reconstructs ne and Te inside the two-power family, so
# no layout brings the RMSE of a plasma state much below the error of the family's best
# fit to its true profile. The fit is scored on each record's own grid with the RMSE
# normalization of the inference, on the first stored run: the truth is the same for
# every layout and noisy repetition.
FIT_UNITS = {
    "electron_density": profile_parametrization_types.DENSITY_UNIT,
    "electron_temperature": profile_parametrization_types.TEMPERATURE_UNIT,
}
floor_rows = []
for set_name, truth_names in (
    ("optimization set", TRUTH_NAMES),
    ("outside the set", TEST_STATES),
):
    per_run = run_placement_inference.accuracy_table(
        run_ids, list(TITLES), truth_names
    )
    state_rmse = (
        run_placement_inference.state_means(per_run[per_run.map_converged])
        .pivot_table(
            index=["truth_name", "profile"], columns="layout", values="error_percent"
        )
    )
    for truth_name in truth_names:
        record = inference_store_utils.load_record(
            inference_store_utils.INFERENCE_STORE,
            run_ids[run_placement_inference.run_label("optimized", truth_name)],
        )
        truth = record.true_profiles
        for profile, unit in FIT_UNITS.items():
            target = np.asarray(
                profile_parametrization_types.from_si(
                    "two_power_zero_edge",
                    np.asarray(getattr(truth, f"{profile}_median")),
                    unit,
                )
            )
            _, fit_error = parametrization_fit_utils.fit_profile(
                "two_power_zero_edge", target, np.asarray(truth.rho)
            )
            floor_rows.append(
                {
                    "set": set_name,
                    "truth": truth_name,
                    "profile": profile,
                    "two-power fit RMSE [%]": fit_error,
                    "initial guess RMSE [%]": state_rmse.loc[
                        (truth_name, profile), "initial_guess"
                    ],
                    "optimized RMSE [%]": state_rmse.loc[(truth_name, profile), "optimized"],
                }
            )
two_power_floor = pd.DataFrame(floor_rows)
two_power_floor.round(3).to_csv(
    SCAN.parent / "if_and_ts_placement_two_power_fit_floor.csv", index=False
)
display(two_power_floor.round(2))
floor_summary = two_power_floor.groupby(["set", "profile"], sort=False)[
    ["two-power fit RMSE [%]", "initial guess RMSE [%]", "optimized RMSE [%]"]
].agg(["mean", "sem"])
floor_summary.round(2)


In [ ]:
# The pressure each plasma state's DESC equilibrium was computed with, at the machine
# scale; the suffix of a state's name is the family it was sampled from.
family_colours = {"parametric": pubfig.PALETTE[0], "gp": pubfig.PALETTE[1]}
state_ids = desc_truth_types.DESC_TRUTH_IDS | desc_truth_types.TEST_STATE_IDS
pressures = pd.read_parquet(
    PARQUET,
    columns=["desc_equilibrium_id", "pressure_knots", "pressure_params"],
    filters=[("desc_equilibrium_id", "in", list(state_ids.values()))],
).set_index("desc_equilibrium_id")
figure, axes = pubfig.subplots(1, 2, width="double", aspect=0.7, sharey=True)
for axis, names in zip(axes, (TRUTH_NAMES, TEST_STATES)):
    for name in names:
        row = pressures.loc[state_ids[name]]
        axis.plot(
            row.pressure_knots,
            1e-6 * reference.machine_scale.pressure_factor * row.pressure_params,
            color=family_colours[name.rsplit("_", 1)[1]],
        )
    axis.set_xlabel(r"$\rho$")
    axis.set_xlim(0.0, 1.0)
axes[0].set_ylim(bottom=0.0)
axes[0].set_ylabel("pressure [MPa]")
axes[0].legend(
    handles=[
        plt.Line2D([], [], color=family_colours["parametric"], label="parametric"),
        plt.Line2D([], [], color=family_colours["gp"], label="Gaussian process"),
    ],
    loc="lower left",
)
pubfig.save(figure, str(PLOTS / "if_and_ts_placement_pressure_profiles"))